In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, bi, dictionary

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat = cfg["catalog"]
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail != "" and not ok else ""))

print(cat, "data dictionary (config/dictionary.yml)")
words = dictionary.load()
check("dictionary.yml loads and every description is one safe line", True)

In [0]:
q = lambda sql: spark.sql(sql).collect()
tables = {(r.table_schema, r.table_name): (r.table_type, r.comment or "") for r in q(f"""
    SELECT table_schema, table_name, table_type, comment FROM {cat}.information_schema.tables
    WHERE table_schema IN ('silver', 'gold')""")}
cols = {}
for r in q(f"""SELECT table_schema, table_name, column_name, comment FROM {cat}.information_schema.columns
               WHERE table_schema = 'gold'"""):
    cols.setdefault(r.table_name, {})[r.column_name] = r.comment or ""

# Silver: table descriptions
silver = words.get("silver", {})
absent = [t for t in silver if ("silver", t) not in tables]
print("INFO silver tables not built here (yet):", absent)
wrong = [t for t, s in silver.items() if ("silver", t) in tables and tables[("silver", t)][1] != s["about"]]
check("every Silver table carries its description", not wrong, wrong)

# Gold: every table documented, every column described
gold = words.get("gold", {})
built = {t for (s, t), (kind, _) in tables.items() if s == "gold" and kind != "VIEW"}
check("every Gold table is in the dictionary", built <= set(gold), sorted(built - set(gold)))
absent = sorted(set(gold) - built)
print("INFO gold tables not built here (yet):", absent)
wrong = [t for t in gold if t in built and tables[("gold", t)][1] != gold[t]["about"]]
check("every Gold table carries its description", not wrong, wrong)
undoc = [f"{t}.{c}" for t in built & set(gold) for c in cols.get(t, {})
         if not c.startswith("_") and c not in gold[t].get("columns", {})]
check("every Gold column is in the dictionary", not undoc, undoc[:20])
stale = [f"{t}.{c}" for t in built & set(gold) for c in gold[t].get("columns", {}) if c not in cols.get(t, {})]
check("no dictionary column that Gold no longer has", not stale, stale[:20])
wrong = [f"{t}.{c}" for t in built & set(gold) for c, text in gold[t].get("columns", {}).items()
         if c in cols.get(t, {}) and cols[t][c] != text]
check("every Gold column carries its description", not wrong, wrong[:20])
n = sum(len(cols.get(t, {})) for t in built & set(gold))
print(f"INFO gold: {len(built & set(gold))} tables, {n} columns described")

# BI views: description from the dictionary
wanted = bi.wanted(cfg)
check("every BI view is in the dictionary", set(wanted) <= set(words.get("views", {})),
      sorted(set(wanted) - set(words.get("views", {}))))
wrong = [v for v in wanted if tables.get(("gold", v), ("", ""))[1] != dictionary.view_comment(v, words)]
check("every BI view carries its description", not wrong, wrong)

print(f"\nt33_dictionary: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t33_dictionary failed")